# Babcock (1961) — The Topology of the Sun's Magnetic Field: Implementation
# 태양 자기장의 위상 구조와 22년 주기: 구현

**Paper / 논문**: H.W. Babcock, "The Topology of the Sun's Magnetic Field and the 22-Year Cycle," *ApJ*, 133, 572–587, 1961.

이 노트북은 Babcock 태양 다이나모 모델의 핵심 물리학을 구현합니다:

1. **Part 1**: 차등 회전에 의한 Toroidal 장 증폭 — Eq. (4)–(8) 재현 / Toroidal field amplification
2. **Part 2**: Spörer 법칙(나비 다이어그램) — Eq. (9) 재현 / Spörer's law (butterfly diagram)
3. **Part 3**: 자기 부력 — Parker 불안정성 / Magnetic buoyancy — Parker instability
4. **Part 4**: Babcock 5단계 모델 시각화 / Five-stage model visualization
5. **Part 5**: 극성 반전 시뮬레이션 — BMR의 f-flux 극이동 / Polarity reversal simulation
6. **Part 6**: 선행 흑점 우세 — flux rope 기하학 / p-spot preponderance — flux rope geometry
7. **Part 7**: 에너지 수지 — 차등 회전 vs. 자기 에너지 / Energy budget

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Circle, Arc, Wedge
from matplotlib.gridspec import GridSpec

plt.rcParams.update({
    'figure.figsize': (14, 7),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

# Solar constants
R_SUN = 6.96e10  # cm
P_EQ = 25.0  # Equatorial rotation period (days)
P_60 = 29.3  # Period at latitude 60° (days)
H0 = 5.0  # Initial field intensity at equator (gauss)
TOTAL_FLUX = 8e21  # Total dipolar flux (maxwells)

## Part 1: Toroidal Field Amplification — Reproducing Eqs. (4)–(8)
## 파트 1: 차등 회전에 의한 Toroidal 장 증폭 — Eq. (4)–(8) 재현

차등 회전율: $\omega = 14°.38 - 2°.77\sin^2\phi$ (Newton & Nunn 1951)

경도 방향 전진량: $\theta = 17.6(n+3)\sin^2\phi$ (rad)

증폭된 자기장: $H \approx 35.2(n+3)H_0\sin\phi$ (when $\psi$ is large)

$H_0 = 5$ G, $n = 0$: $H = 528\sin\phi$, 위도 30°에서 $H = 264$ G (임계값).

Differential rotation rate: $\omega = 14°.38 - 2°.77\sin^2\phi$.
Amplified field: $H \approx 35.2(n+3)H_0\sin\phi$.
At $H_0 = 5$ G, $n = 0$: $H = 528\sin\phi$, giving 264 G at lat 30° (critical).

In [ ]:
def omega_deg_per_day(phi_deg: np.ndarray) -> np.ndarray:
    """Differential rotation rate (Newton & Nunn 1951, Eq. 1).

    Args:
        phi_deg: Latitude in degrees.

    Returns:
        Angular velocity in degrees/day.
    """
    return 14.38 - 2.77 * np.sin(np.radians(phi_deg))**2


def theta_advance(n: float, phi_deg: np.ndarray) -> np.ndarray:
    """Longitudinal advance in radians (Eq. 4).

    Args:
        n: Years since beginning of new sunspot cycle.
        phi_deg: Latitude in degrees.

    Returns:
        Longitudinal advance in radians.
    """
    return 17.6 * (n + 3) * np.sin(np.radians(phi_deg))**2


def tan_psi(n: float, phi_deg: np.ndarray) -> np.ndarray:
    """Tangent of spiral angle (Eq. 5).

    Args:
        n: Years since cycle onset.
        phi_deg: Latitude in degrees.

    Returns:
        tan(psi) — spiral angle tangent.
    """
    phi = np.radians(phi_deg)
    return 35.2 * (n + 3) * np.sin(phi) * np.cos(phi)


def H_amplified(n: float, phi_deg: np.ndarray, H0: float = 5.0) -> np.ndarray:
    """Amplified toroidal field intensity (Eq. 7, large-psi approximation).

    Args:
        n: Years since cycle onset.
        phi_deg: Latitude in degrees.
        H0: Initial equatorial field (gauss).

    Returns:
        Amplified field in gauss.
    """
    return 35.2 * (n + 3) * H0 * np.sin(np.radians(phi_deg))


phi = np.linspace(2, 55, 200)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# --- Top Left: Differential rotation ---
ax = axes[0, 0]
ax.plot(phi, omega_deg_per_day(phi), 'b-', lw=2.5)
ax.axhline(omega_deg_per_day(0), color='red', ls='--', lw=1, label=f'Equator: {omega_deg_per_day(0):.2f}°/day')
ax.axhline(omega_deg_per_day(60), color='green', ls='--', lw=1, label=f'60°: {omega_deg_per_day(60):.2f}°/day')
ax.set_xlabel('Latitude φ (°)', fontsize=12)
ax.set_ylabel('ω (°/day)', fontsize=12)
ax.set_title('Differential Rotation Rate (Eq. 1)\n차등 회전율', fontsize=13)
ax.legend(fontsize=10)

# --- Top Right: Longitudinal advance θ ---
ax = axes[0, 1]
for n in [0, 2, 5, 8]:
    ax.plot(phi, theta_advance(n, phi) / (2*np.pi), lw=2,
            label=f'n = {n} yr (total {n+3} yr)')
ax.set_xlabel('Latitude φ (°)', fontsize=12)
ax.set_ylabel('Extra turns relative to pole', fontsize=12)
ax.set_title('Longitudinal Advance θ/(2π) (Eq. 4)\n경도 전진량 (바퀴 수)', fontsize=13)
ax.legend(fontsize=10)

# --- Bottom Left: Amplified field H ---
ax = axes[1, 0]
for n in [0, 2, 5, 8]:
    H = H_amplified(n, phi)
    ax.plot(phi, H, lw=2, label=f'n = {n} yr')
ax.axhline(264, color='red', ls='--', lw=2, label='Critical H = 264 G')
ax.axhline(528, color='darkred', ls=':', lw=1.5, label='Max (n=0) = 528 G')
ax.set_xlabel('Latitude φ (°)', fontsize=12)
ax.set_ylabel('H (gauss)', fontsize=12)
ax.set_title('Amplified Toroidal Field (Eq. 7)\n증폭된 Toroidal 자기장', fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, 1200)

# --- Bottom Right: Spiral angle ψ ---
ax = axes[1, 1]
for n in [0, 2, 5, 8]:
    psi_deg = np.degrees(np.arctan(tan_psi(n, phi)))
    ax.plot(phi, psi_deg, lw=2, label=f'n = {n} yr')
ax.axhline(90, color='gray', ls=':', lw=1)
ax.set_xlabel('Latitude φ (°)', fontsize=12)
ax.set_ylabel('Spiral angle ψ (°)', fontsize=12)
ax.set_title('Spiral Angle (Eq. 5)\n나선 각도', fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(0, 95)

plt.suptitle("Babcock Model: Stage 2 — Toroidal Field Amplification\n"
             "Babcock 모델: 2단계 — Toroidal 장 증폭", fontsize=15, y=1.02)
plt.tight_layout()
plt.show()

# Print key values
print("Key values from the paper / 논문의 핵심 수치:")
print(f"  At n=0, φ=30°: H = {H_amplified(0, 30):.0f} G (paper: 264 G) ✓")
print(f"  At n=0, φ=90°: H = {H_amplified(0, 90):.0f} G (paper: 528 G) ✓")
print(f"  Equator gains {theta_advance(0, 55)/(2*np.pi):.1f} extra turns in 3 years"
      f" (paper: 5.6) ✓")

## Part 2: Spörer's Law — Reproducing Fig. 4 (Butterfly Diagram)
## 파트 2: Spörer 법칙 — Fig. 4 재현 (나비 다이어그램)

Babcock의 핵심 결과: $\sin\phi = \pm\frac{1.5}{n+3}$ (Eq. 9)

이 공식은 임계 자기장 세기(264 G)에 도달하는 위도를 시간의 함수로 나타냅니다.
합성 흑점 데이터를 겹쳐 나비 다이어그램을 생성합니다.

Babcock's key result: $\sin\phi = \pm 1.5/(n+3)$ — the latitude where the critical field (264 G) is reached as a function of time. We overlay synthetic sunspot data to create a butterfly diagram.

In [ ]:
def sporer_latitude(n: np.ndarray) -> np.ndarray:
    """Spörer's law: critical latitude as function of time (Eq. 9).

    Args:
        n: Years since beginning of sunspot cycle.

    Returns:
        Latitude in degrees where critical field is first reached.
    """
    sin_phi = 1.5 / (n + 3)
    sin_phi = np.clip(sin_phi, 0, 1)
    return np.degrees(np.arcsin(sin_phi))


# Reproduce Fig. 4 from the paper
fig, ax = plt.subplots(figsize=(16, 8))

# Babcock's Eq. (9) curve
n_range = np.linspace(-2, 12, 500)
phi_crit = sporer_latitude(n_range)

# Plot for two consecutive cycles
cycle_start_1 = 1953  # Cycle 19 start (Babcock's reference)
cycle_start_2 = cycle_start_1 + 11

for cycle_start, cycle_num in [(cycle_start_1, '19'), (cycle_start_2, '20')]:
    years = n_range + cycle_start
    # North hemisphere
    ax.plot(years, phi_crit, 'b-', lw=2.5,
            label=f'Eq.(9): sin φ = 1.5/(n+3)' if cycle_start == cycle_start_1 else '')
    # South hemisphere
    ax.plot(years, -phi_crit, 'b-', lw=2.5)

# Stage markers from Fig. 4
ax.axvline(1953, color='gray', ls='--', lw=1, alpha=0.5)
ax.axvline(1955, color='green', ls='--', lw=1.5, alpha=0.7)
ax.axvline(1964, color='gray', ls='--', lw=1, alpha=0.5)

# Add synthetic sunspot scatter
rng = np.random.default_rng(42)
for cycle_start in [cycle_start_1, cycle_start_2]:
    n_spots = 600
    n_vals = rng.uniform(0, 9, n_spots)
    phi_max = sporer_latitude(n_vals)
    phi_spots = rng.normal(0, 1) + phi_max * rng.uniform(0.3, 1.0, n_spots)
    phi_spots = np.clip(phi_spots, 2, 40)
    hemisphere = rng.choice([-1, 1], n_spots)
    years_spots = n_vals + cycle_start

    # Spot density proportional to solar cycle (sine-like)
    cycle_phase = n_vals / 11.0
    weight = np.sin(np.pi * cycle_phase)**1.5
    keep = rng.random(n_spots) < weight
    ax.scatter(years_spots[keep], phi_spots[keep] * hemisphere[keep],
               s=3, alpha=0.3, c='orange', zorder=1)

# Annotations
ax.text(1954, 32, 'STAGE 1\n(Initial dipole)', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', fc='lightyellow'))
ax.text(1957, -32, 'STAGE 2\nAMPLIFICATION', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', fc='lightyellow'))
ax.text(1960, 25, "BMR's ⊙⊙", fontsize=10, ha='center',
        bbox=dict(boxstyle='round', fc='lightyellow'))
ax.text(1965, -28, 'STAGE 5\n(Reversed dipole)', fontsize=10, ha='center',
        bbox=dict(boxstyle='round', fc='lightyellow'))

# Formatting
ax.set_xlabel('Year', fontsize=13)
ax.set_ylabel('Latitude φ (°)', fontsize=13)
ax.set_title("Babcock's Fig. 4: Spörer's Law — sin φ = ±1.5/(n+3)\n"
             "Babcock의 Fig. 4: Spörer 법칙 — 나비 다이어그램", fontsize=14)
ax.axhline(0, color='black', lw=1)
ax.set_xlim(1951, 1979)
ax.set_ylim(-40, 40)
ax.legend(fontsize=11, loc='upper right')

plt.tight_layout()
plt.show()

# Print Spörer law values
print("Spörer's Law (Eq. 9): sin φ = ±1.5/(n+3)")
print("-" * 40)
for n in range(0, 10):
    print(f"  n = {n} yr: φ = {sporer_latitude(n):.1f}°")

## Part 3: Magnetic Buoyancy — Parker Instability (Eq. 10)
## 파트 3: 자기 부력 — Parker 불안정성

$p_e = p_i + \frac{H^2}{8\pi}$ → $p_i < p_e$ → $\rho_i < \rho_e$ → 부력!

Flux tube가 떠오르려면 자기압($H^2/8\pi$)이 외부 가스압의 유의미한 비율이어야 합니다.
플라즈마 $\beta = 8\pi p / H^2$ 파라미터로 이를 정량화합니다.

For a flux tube to rise, magnetic pressure ($H^2/8\pi$) must be a significant fraction of external gas pressure. Quantified via plasma $\beta = 8\pi p / H^2$.

In [ ]:
def magnetic_pressure_cgs(H_gauss: np.ndarray) -> np.ndarray:
    """Magnetic pressure in CGS (dyne/cm²).

    Args:
        H_gauss: Field strength in gauss.

    Returns:
        Magnetic pressure H²/(8π) in dyne/cm².
    """
    return H_gauss**2 / (8 * np.pi)


def density_deficit(H_gauss: float, T: float, mu_mol: float = 0.6) -> float:
    """Fractional density deficit inside a flux tube (isothermal).

    Assumes pressure balance p_e = p_i + H²/8π with same temperature.
    From ideal gas: Δρ/ρ = (p_e - p_i)/p_e = (H²/8π)/p_e.

    Args:
        H_gauss: Field strength inside tube (gauss).
        T: Temperature (K).
        mu_mol: Mean molecular weight.

    Returns:
        Fractional density deficit Δρ/ρ.
    """
    k_B = 1.38e-16  # erg/K
    m_H = 1.67e-24  # g
    # Estimate external pressure from temperature and density
    # At depth ~0.1 R_sun: T ~ 2e6 K, n ~ 1e17 cm^-3
    n_e = 1e17  # number density cm^-3
    p_e = n_e * k_B * T  # gas pressure
    p_mag = magnetic_pressure_cgs(H_gauss)
    return p_mag / p_e


fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# --- Left: Pressure balance diagram ---
ax = axes[0]
H_range = np.linspace(0, 3000, 500)
p_mag = magnetic_pressure_cgs(H_range)

# Typical external gas pressures at different depths
p_photosphere = 1e5  # dyne/cm² (photosphere)
p_shallow = 1e8  # dyne/cm² (shallow interior ~0.05 R)
p_deep = 1e12  # dyne/cm² (0.1 R depth)

ax.plot(H_range, p_mag, 'b-', lw=2.5, label='$H^2/8\\pi$ (magnetic)')
ax.axhline(p_photosphere, color='orange', ls='--', lw=2, label=f'Photosphere: {p_photosphere:.0e}')
ax.axhline(p_shallow, color='red', ls='--', lw=2, label=f'0.05 R depth: {p_shallow:.0e}')
ax.axhline(p_deep, color='darkred', ls='--', lw=2, label=f'0.1 R depth: {p_deep:.0e}')
ax.axvline(264, color='green', ls=':', lw=2, label='Babcock critical: 264 G')
ax.axvline(1000, color='purple', ls=':', lw=2, label='Flux rope: ~1 kG')
ax.set_xlabel('Field Strength H (gauss)', fontsize=12)
ax.set_ylabel('Pressure (dyne/cm²)', fontsize=12)
ax.set_yscale('log')
ax.set_title('Magnetic vs Gas Pressure\n자기압 vs 가스압', fontsize=13)
ax.legend(fontsize=8, loc='lower right')

# --- Middle: Buoyancy force schematic ---
ax = axes[1]
ax.set_xlim(-2, 2)
ax.set_ylim(-2, 3)
ax.set_aspect('equal')
ax.axis('off')

# External medium
ax.add_patch(plt.Rectangle((-2, -2), 4, 5, fc='lightyellow', ec='none', alpha=0.5))
ax.text(0, 2.7, 'External plasma\n외부 플라즈마', ha='center', fontsize=11)
ax.text(-1.5, 2.2, f'$p_e$, $\\rho_e$, T', fontsize=12)

# Flux tube cross-section
circle = Circle((0, 0.5), 0.8, fc='lightblue', ec='blue', lw=3)
ax.add_patch(circle)
ax.text(0, 0.5, '$p_i + \\frac{H^2}{8\\pi} = p_e$\n\n$p_i < p_e$\n$\\rho_i < \\rho_e$',
        ha='center', va='center', fontsize=10)

# B field (into page)
for dx, dy in [(-0.3, 0.3), (0.3, 0.3), (0, 0.7), (-0.3, 0.7), (0.3, 0.7)]:
    ax.plot(dx, 0.5 + dy - 0.2, 'x', color='blue', ms=8, mew=2)

# Buoyancy arrow
ax.annotate('', xy=(0, 2.0), xytext=(0, 1.5),
            arrowprops=dict(arrowstyle='->', color='red', lw=3))
ax.text(0.5, 1.8, 'Buoyancy\n부력', fontsize=12, color='red', fontweight='bold')

# Gravity arrow
ax.annotate('', xy=(0, -0.8), xytext=(0, -0.3),
            arrowprops=dict(arrowstyle='->', color='gray', lw=2))
ax.text(0.5, -0.6, 'Gravity\n중력', fontsize=10, color='gray')

ax.set_title('Flux Tube Cross-Section\nFlux Tube 단면', fontsize=13, pad=15)

# --- Right: Plasma beta ---
ax = axes[2]
H_range2 = np.logspace(1, 4, 200)
T_values = [5e5, 2e6, 5e6]
n_e = 1e17  # typical for shallow interior

for T in T_values:
    p_gas = n_e * 1.38e-16 * T
    beta = 8 * np.pi * p_gas / H_range2**2
    ax.plot(H_range2, beta, lw=2, label=f'T = {T:.0e} K')

ax.axhline(1, color='red', ls='--', lw=2, label='β = 1 (equipartition)')
ax.fill_between(H_range2, 0, 1, alpha=0.1, color='red')
ax.text(2000, 0.3, 'Magnetically\ndominated\n(buoyant!)', fontsize=10,
        ha='center', color='red')
ax.text(50, 100, 'Gas\ndominated', fontsize=10, ha='center', color='blue')

ax.set_xlabel('Field Strength H (gauss)', fontsize=12)
ax.set_ylabel('Plasma β = 8πp/H²', fontsize=12)
ax.set_xscale('log')
ax.set_yscale('log')
ax.set_ylim(0.01, 1e4)
ax.set_title('Plasma Beta\n($n_e = 10^{17}$ cm$^{-3}$)', fontsize=13)
ax.legend(fontsize=9)

plt.suptitle("Magnetic Buoyancy: Why Flux Tubes Rise (Stage 3)\n"
             "자기 부력: Flux Tube가 떠오르는 이유 (3단계)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("At H = 1000 G (flux rope), T = 2×10⁶ K, n = 10¹⁷ cm⁻³:")
print(f"  Magnetic pressure: {magnetic_pressure_cgs(1000):.1e} dyne/cm²")
print(f"  Gas pressure: {1e17 * 1.38e-16 * 2e6:.1e} dyne/cm²")
print(f"  β = {8*np.pi*1e17*1.38e-16*2e6 / 1000**2:.1f}")
print("  → β ~ 1: buoyancy is significant! ✓")

## Part 4: Babcock's Five-Stage Model — Schematic Visualization
## 파트 4: Babcock 5단계 모델 — 도식적 시각화

5단계를 한눈에 보여주는 도식: 태양 단면에서의 자기장 진화.

Schematic showing all 5 stages: magnetic field evolution in a solar cross-section.

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 5))

stage_info = [
    ("Stage 1\nInitial Dipole\n초기 쌍극자장\n(Solar minimum)", 'lightyellow'),
    ("Stage 2\nAmplification\n차등 회전 증폭\n(+3 years)", 'lightyellow'),
    ("Stage 3\nBMR Formation\nBMR 형성\n(Spots appear)", 'lightyellow'),
    ("Stage 4\nNeutralization\n극성 중화/반전\n(Max → decline)", 'lightyellow'),
    ("Stage 5\nReversed Dipole\n반전된 쌍극자장\n(New minimum)", 'lightyellow'),
]

for idx, (ax, (title, fc)) in enumerate(zip(axes, stage_info)):
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    ax.set_aspect('equal')
    ax.axis('off')

    # Draw sun
    sun = Circle((0, 0), 1.0, fc='#fff8dc', ec='orange', lw=2)
    ax.add_patch(sun)

    # Equator
    ax.plot([-1, 1], [0, 0], 'k-', lw=0.5, alpha=0.3)

    if idx == 0:  # Stage 1: Poloidal dipole
        # Dipole field lines
        theta = np.linspace(0, np.pi, 50)
        for r_scale in [0.4, 0.7, 1.0, 1.3]:
            x = r_scale * np.sin(theta)**2 * np.cos(theta) * 2
            y = r_scale * np.sin(theta)**2 * np.sin(theta) * 2 - r_scale
            mask = x**2 + (y + r_scale - r_scale * 0.5)**2 > 0
            ax.plot(x * 0.5, (y + r_scale) * 0.7 - 0.2, 'b-', lw=0.8, alpha=0.5)
        ax.text(0, 1.15, 'N (+)', fontsize=9, ha='center', color='red')
        ax.text(0, -1.15, 'S (−)', fontsize=9, ha='center', color='blue')

    elif idx == 1:  # Stage 2: Winding
        # Toroidal field (wrapped lines)
        for lat in [0.2, 0.4, 0.6]:
            for sign in [1, -1]:
                y = sign * lat
                width = np.sqrt(1 - y**2) * 0.9
                t = np.linspace(-width, width, 50)
                wiggle = 0.03 * np.sin(8 * np.pi * t / width)
                ax.plot(t, y + wiggle, 'b-', lw=1.0, alpha=0.6)
        # Arrows showing rotation direction
        ax.annotate('', xy=(0.6, 0.05), xytext=(-0.6, 0.05),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1.5))
        ax.annotate('', xy=(0.3, 0.55), xytext=(-0.3, 0.55),
                    arrowprops=dict(arrowstyle='->', color='red', lw=1))
        ax.text(0, -0.85, 'Ω fast', fontsize=8, ha='center', color='red')
        ax.text(0, 0.75, 'Ω slow', fontsize=8, ha='center', color='red')

    elif idx == 2:  # Stage 3: BMR emergence
        # Toroidal field background
        for lat in [0.2, 0.4]:
            for sign in [1, -1]:
                y = sign * lat
                width = np.sqrt(1 - y**2) * 0.8
                ax.plot([-width, width], [y, y], 'b-', lw=0.8, alpha=0.3)
        # Ω-loops rising
        for (x0, y0, s) in [(0.3, 0.5, 1), (-0.3, -0.4, -1)]:
            t = np.linspace(-0.2, 0.2, 50)
            loop_y = y0 + 0.3 * (1 - (t/0.2)**2)
            ax.plot(x0 + t, loop_y, 'r-', lw=2.5)
            ax.plot(x0 - 0.15, y0 + 0.02, 'ro', ms=6)  # p spot
            ax.plot(x0 + 0.15, y0 + 0.02, 'bo', ms=5)  # f spot
            ax.text(x0 - 0.15, y0 - 0.12, 'p', fontsize=7, ha='center', color='red')
            ax.text(x0 + 0.15, y0 - 0.12, 'f', fontsize=7, ha='center', color='blue')

    elif idx == 3:  # Stage 4: Neutralization
        # f-flux migrating poleward
        for sign in [1, -1]:
            ax.annotate('', xy=(0.2, sign * 0.9), xytext=(0.4, sign * 0.4),
                        arrowprops=dict(arrowstyle='->', color='blue', lw=2))
            ax.text(0.55, sign * 0.6, 'f flux\n→ pole', fontsize=7, color='blue')
        # p-flux equatorward
        for sign in [1, -1]:
            ax.annotate('', xy=(-0.2, sign * 0.1), xytext=(-0.4, sign * 0.4),
                        arrowprops=dict(arrowstyle='->', color='red', lw=2))
        ax.text(-0.6, 0, 'p flux\n→ equator', fontsize=7, color='red', ha='center')
        # Reconnection symbol
        ax.text(0, 1.15, 'N: neutralizing', fontsize=8, ha='center', color='purple')

    elif idx == 4:  # Stage 5: Reversed dipole
        # Same as Stage 1 but reversed
        theta = np.linspace(0, np.pi, 50)
        for r_scale in [0.4, 0.7, 1.0, 1.3]:
            x = r_scale * np.sin(theta)**2 * np.cos(theta) * 2
            y = r_scale * np.sin(theta)**2 * np.sin(theta) * 2 - r_scale
            ax.plot(x * 0.5, (y + r_scale) * 0.7 - 0.2, 'r-', lw=0.8, alpha=0.5)
        ax.text(0, 1.15, 'N (−)', fontsize=9, ha='center', color='blue')
        ax.text(0, -1.15, 'S (+)', fontsize=9, ha='center', color='red')
        ax.text(0, -1.35, 'REVERSED!', fontsize=8, ha='center', color='darkred',
                fontweight='bold')

    ax.set_title(title, fontsize=10, pad=10)

    # Arrow between stages
    if idx < 4:
        axes[idx].annotate('', xy=(1.45, 0), xytext=(1.25, 0),
                           arrowprops=dict(arrowstyle='->', color='green', lw=2.5))

plt.suptitle("Babcock's Five-Stage Solar Dynamo Model\n"
             "Babcock의 5단계 태양 다이나모 모델",
             fontsize=15, y=1.05)
plt.tight_layout()
plt.show()

print("Stage 1 → 2: Differential rotation winds poloidal into toroidal (~3 yr)")
print("Stage 2 → 3: Magnetic buoyancy lifts flux ropes → BMRs at surface")
print("Stage 3 → 4: Joy's law tilt → f-flux poleward → neutralizes polar field")
print("Stage 4 → 5: Reversed dipole established (~11 yr total)")
print("Stage 5 → 1: Process repeats → 22-year full magnetic cycle")

## Part 5: Polarity Reversal Simulation — 1D Flux Transport
## 파트 5: 극성 반전 시뮬레이션 — 1D Flux Transport

BMR의 후행(f) 극성 flux가 극 방향으로 이동하여 극 자기장을 반전시키는 과정을
1D 확산-이류(diffusion-advection) 방정식으로 시뮬레이션합니다.

$$\frac{\partial B_r}{\partial t} = -\frac{1}{R\sin\theta}\frac{\partial}{\partial\theta}(v_\theta B_r \sin\theta) + \frac{\eta}{R^2\sin\theta}\frac{\partial}{\partial\theta}\left(\sin\theta\frac{\partial B_r}{\partial\theta}\right) + S(\theta, t)$$

여기서 $v_\theta$는 자오선 흐름, $\eta$는 확산 계수, $S$는 BMR 소스.

Simulating poleward migration of f-polarity flux using a 1D diffusion-advection equation.

In [ ]:
def simulate_polar_reversal(N_lat: int = 180, N_years: float = 22,
                            dt_days: float = 5.0, eta_km2s: float = 250.0,
                            v_merid_ms: float = 15.0) -> dict:
    """Simulate polar field reversal via 1D surface flux transport.

    Args:
        N_lat: Number of latitude grid points.
        N_years: Total simulation time in years.
        dt_days: Time step in days.
        eta_km2s: Turbulent diffusivity (km²/s).
        v_merid_ms: Peak meridional flow speed (m/s).

    Returns:
        Dictionary with latitude, time, and Br evolution.
    """
    R = 6.96e5  # Solar radius in km
    theta = np.linspace(0.01, np.pi - 0.01, N_lat)  # Colatitude
    lat = 90 - np.degrees(theta)  # Latitude
    dtheta = theta[1] - theta[0]

    eta = eta_km2s  # km²/s
    dt = dt_days * 86400 / (R**2)  # Normalized time step

    # Initial polar field: dipole-like
    Br = 2.0 * np.cos(theta)  # Simple dipole: positive at north pole

    # Meridional flow: poleward in both hemispheres
    v_theta = -v_merid_ms * 1e-3 / R * np.sin(2 * theta)  # Normalized

    # BMR source: Joy's law tilted bipolar sources at mid-latitudes
    N_steps = int(N_years * 365.25 / dt_days)
    snapshot_years = [0, 2, 4, 6, 8, 10, 11, 14, 18, 22]
    snapshots = []
    polar_north = []
    polar_south = []
    times_yr = []

    for step in range(N_steps):
        t_yr = step * dt_days / 365.25

        # Record snapshots
        for sy in snapshot_years:
            if abs(t_yr - sy) < dt_days / 365.25 and not any(
                abs(s['t'] - sy) < 0.1 for s in snapshots
            ):
                snapshots.append({'t': t_yr, 'Br': Br.copy(), 'lat': lat.copy()})

        # Polar field strength (average above 70° latitude)
        north_mask = lat > 70
        south_mask = lat < -70
        if np.any(north_mask):
            polar_north.append(np.mean(Br[north_mask]))
        if np.any(south_mask):
            polar_south.append(np.mean(Br[south_mask]))
        times_yr.append(t_yr)

        # BMR source: active during years 0-8 of each 11-year half-cycle
        cycle_phase = t_yr % 11.0
        if 1 < cycle_phase < 9:
            # BMR emergence at latitude given by Spörer's law
            n = cycle_phase
            phi_spot = sporer_latitude(n)
            theta_spot_N = np.radians(90 - phi_spot)
            theta_spot_S = np.radians(90 + phi_spot)

            # Source amplitude (sine-shaped activity curve)
            activity = np.sin(np.pi * (cycle_phase - 1) / 8)**2 * 0.3

            # Joy's law: f-polarity slightly poleward, p-polarity equatorward
            tilt = 5.0  # degrees of tilt
            sigma_bmr = np.radians(5.0)

            # Determine polarity based on cycle half
            cycle_half = int(t_yr / 11.0)
            pol = 1 if cycle_half % 2 == 0 else -1

            # North: f(+pol) at higher lat, p(-pol) at lower lat
            S_N = activity * pol * (
                np.exp(-((theta - theta_spot_N + np.radians(tilt))**2) / (2*sigma_bmr**2))
                - np.exp(-((theta - theta_spot_N - np.radians(tilt))**2) / (2*sigma_bmr**2))
            )
            # South: opposite
            S_S = activity * (-pol) * (
                np.exp(-((theta - theta_spot_S - np.radians(tilt))**2) / (2*sigma_bmr**2))
                - np.exp(-((theta - theta_spot_S + np.radians(tilt))**2) / (2*sigma_bmr**2))
            )

            Br += (S_N + S_S) * dt_days / 365.25

        # Diffusion term
        sin_theta = np.sin(theta)
        d2Br = np.zeros_like(Br)
        for i in range(1, N_lat - 1):
            d2Br[i] = (1.0 / (sin_theta[i] * dtheta**2)) * (
                sin_theta[i+1] * (Br[i+1] - Br[i]) / dtheta
                - sin_theta[i-1] * (Br[i] - Br[i-1]) / dtheta
            ) / (R**2)
        Br += eta * d2Br * dt * R**2

        # Advection term (meridional flow)
        dFdtheta = np.zeros_like(Br)
        flux = v_theta * Br * sin_theta
        for i in range(1, N_lat - 1):
            dFdtheta[i] = (flux[i+1] - flux[i-1]) / (2 * dtheta)
        Br -= dFdtheta / sin_theta * dt * R

        # Boundary conditions
        Br[0] = Br[1]
        Br[-1] = Br[-2]

    return {
        'snapshots': snapshots,
        'polar_north': np.array(polar_north),
        'polar_south': np.array(polar_south),
        'times_yr': np.array(times_yr),
    }


result = simulate_polar_reversal()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# --- Top: Latitude-time snapshots ---
colors = plt.cm.viridis(np.linspace(0, 0.9, len(result['snapshots'])))
for snap, color in zip(result['snapshots'], colors):
    ax1.plot(snap['lat'], snap['Br'], color=color, lw=1.5,
             label=f"t = {snap['t']:.0f} yr")
ax1.axhline(0, color='gray', ls='-', lw=0.5)
ax1.set_xlabel('Latitude (°)', fontsize=12)
ax1.set_ylabel('$B_r$ (normalized)', fontsize=12)
ax1.set_title('Surface Radial Field Evolution\n표면 방사 자기장 진화', fontsize=13)
ax1.legend(fontsize=8, ncol=5)

# --- Bottom: Polar field strength over time ---
ax2.plot(result['times_yr'], result['polar_north'], 'r-', lw=2.5,
         label='North pole (> 70°)')
ax2.plot(result['times_yr'], result['polar_south'], 'b-', lw=2.5,
         label='South pole (< −70°)')
ax2.axhline(0, color='gray', ls='--', lw=1)

# Mark reversal times
for t_rev in [5, 16]:
    ax2.axvline(t_rev, color='green', ls=':', lw=1.5, alpha=0.5)
ax2.text(5.5, max(result['polar_north']) * 0.8, 'Reversal\n반전', fontsize=10,
         color='green')

ax2.set_xlabel('Time (years)', fontsize=12)
ax2.set_ylabel('Polar Field (normalized)', fontsize=12)
ax2.set_title('Polar Field Reversal — Babcock Stage 4\n극 자기장 반전', fontsize=13)
ax2.legend(fontsize=11)

plt.suptitle("Polarity Reversal via Surface Flux Transport\n"
             "표면 자기 flux 수송에 의한 극성 반전", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("The polar field reverses near solar maximum (~5 yr into cycle)")
print("This matches Babcock's Stage 4 and observational evidence")

## Part 6: Efficiency Calculation — Why Only 1% Is Needed
## 파트 6: 효율 계산 — 왜 1%만 필요한가

Babcock의 핵심 통찰: 극성 반전에 필요한 flux는 BMR이 생산하는 총 flux의 단 ~1%이다.

Babcock's key insight: only ~1% of total BMR flux is needed for polarity reversal.

- 반전에 필요한 flux: $2 \times 8 \times 10^{21} = 1.6 \times 10^{22}$ Mx
- 한 주기의 총 BMR flux: $\sim 3000 \times 10^{21} = 3 \times 10^{24}$ Mx
- 효율: $1.6 \times 10^{22} / 3 \times 10^{24} \approx 0.5\%$

In [ ]:
# Babcock's efficiency calculation
dipole_flux = 8e21  # Mx (one hemisphere)
flux_for_reversal = 2 * dipole_flux  # Need to cancel AND rebuild

n_bmr_per_cycle = 3000  # Observed spot groups per cycle
flux_per_bmr = 1e21  # Mx per typical BMR
total_bmr_flux = n_bmr_per_cycle * flux_per_bmr

efficiency = flux_for_reversal / total_bmr_flux * 100

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Left: Flux budget ---
ax = axes[0]
categories = ['Dipole flux\n(one hemisphere)\n쌍극자 flux', 
              'Flux for reversal\n(cancel + rebuild)\n반전에 필요한 flux',
              'Total BMR flux\n(per cycle)\n주기당 총 BMR flux']
values = [dipole_flux, flux_for_reversal, total_bmr_flux]
colors = ['steelblue', 'orange', 'green']

bars = ax.barh(categories, values, color=colors, edgecolor='black')
ax.set_xscale('log')
ax.set_xlabel('Magnetic Flux (Mx)', fontsize=12)
ax.set_title('Flux Budget / 자기 flux 수지', fontsize=13)

for bar, val in zip(bars, values):
    ax.text(val * 1.5, bar.get_y() + bar.get_height()/2,
            f'{val:.0e} Mx', va='center', fontsize=11)

# --- Right: Efficiency vs cycle amplitude ---
ax = axes[1]
# What if the cycle is stronger or weaker?
n_bmr_range = np.linspace(500, 5000, 100)
eff = flux_for_reversal / (n_bmr_range * flux_per_bmr) * 100

ax.plot(n_bmr_range, eff, 'b-', lw=2.5)
ax.axhline(1, color='red', ls='--', lw=2, label='1% efficiency')
ax.axvline(3000, color='green', ls=':', lw=2, label='Typical cycle (~3000 BMRs)')
ax.fill_between(n_bmr_range, 0, eff, alpha=0.1, color='blue')

# Mark weak and strong cycles
ax.axvline(1000, color='orange', ls=':', lw=1.5, label='Weak cycle (Maunder?)')
ax.axvline(4500, color='purple', ls=':', lw=1.5, label='Strong cycle')

ax.set_xlabel('Number of BMRs per cycle', fontsize=12)
ax.set_ylabel('Required efficiency (%)', fontsize=12)
ax.set_title('Reversal Efficiency vs Cycle Strength\n반전 효율 vs 주기 강도', fontsize=13)
ax.legend(fontsize=9)
ax.set_ylim(0, 5)

plt.suptitle("Babcock's ~1% Efficiency for Polarity Reversal\n"
             "극성 반전에 필요한 ~1% 효율", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"Efficiency calculation:")
print(f"  Dipole flux: {dipole_flux:.0e} Mx")
print(f"  Flux needed for reversal: {flux_for_reversal:.0e} Mx")
print(f"  Total BMR flux available: {total_bmr_flux:.0e} Mx")
print(f"  Required efficiency: {efficiency:.1f}%")
print(f"\nImplication: Process is ROBUST (always works) but SENSITIVE")
print(f"  (small variations → large cycle-to-cycle differences)")

## Summary / 요약

| Part | 내용 / Content | Babcock (1961) 연결 / Connection |
|---|---|---|
| 1 | 차등 회전 + Toroidal 증폭 (Eqs. 4–8) | Stage 2 — poloidal → toroidal 변환, 증폭 인자 재현 |
| 2 | Spörer 법칙 (Eq. 9) + 나비 다이어그램 | Fig. 4 재현 — $\sin\phi = \pm 1.5/(n+3)$ |
| 3 | 자기 부력 (Eq. 10) + 플라즈마 β | Stage 3 — flux tube 부상의 물리적 조건 |
| 4 | 5단계 모델 도식 | Stages 1–5 전체 순환 — Fig. 1, 2, 3, 8 요약 |
| 5 | 극성 반전 시뮬레이션 (1D flux transport) | Stage 4 — f-flux 극이동에 의한 극성 반전 |
| 6 | 효율 계산 (~1%) | 극성 반전의 robustness와 sensitivity |

**다음 논문 / Next paper**: Leighton (1969) — 운동학적 다이나모 모델.
Babcock의 현상학적 모델을 **편미분방정식**으로 정량화합니다. 초립자 확산과 자오선 흐름을
도입하여 flux transport를 수학적으로 기술 → "Babcock-Leighton dynamo"의 완성.

**Next paper**: Leighton (1969) — Kinematic dynamo model.
Quantifies Babcock's phenomenological model into **PDEs** with supergranular diffusion and
meridional flow → completing the "Babcock-Leighton dynamo."